# 🧭 A Tour of Numerical Methods with CDS

An interactive walkthrough of classical numerical algorithms — implemented in **readable pure Python** with no NumPy/SciPy.

> ▶️ **You're in Binder.** Every cell below is editable. Hit `Shift+Enter` to run a cell, then experiment with the parameters. This notebook mirrors the [Tour of Numerical Methods](https://Furox88.github.io/cognitive-discovery-system/tour_of_numerical_methods/) guide in the docs.

In [ ]:
import cds

print("Cognitive Discovery System v" + cds.__version__)

## 1 · Numerical Integration — converging to the truth

We integrate $\int_0^1 e^x\,dx = e - 1 \approx 1.7182818...$ with a ladder of rules, each more accurate per function-evaluation than the last. Watch the error shrink as the rules get smarter.

In [ ]:
import math

from cds.numerical_integration import gaussian_quadrature, romberg, simpson, trapezoid

f = math.exp
true_value = math.e - 1

rules = {
    "Trapezoid (n=100)": trapezoid(f, 0, 1, n=100),
    "Simpson 1/3 (n=100)": simpson(f, 0, 1, n=100),
    "Romberg (max_iter=64)": romberg(f, 0, 1, max_iter=64).value,
    "Gauss-Legendre (n=5)": gaussian_quadrature(f, 0, 1, n=5),
}

print(f"{'Rule':<25} {'Estimate':>12} {'Error':>14}")
print("-" * 53)
for name, estimate in rules.items():
    print(f"{name:<25} {estimate:>12.8f} {estimate - true_value:>+14.2e}")
print(f"{'Exact (e-1)':<25} {true_value:>12.8f}")

**Takeaway:** Gauss-Legendre with just **5 evaluations** beats trapezoid's 100 — algorithmic intelligence beats brute force. This is the core thesis of CDS.

## 2 · Monte Carlo — intelligence from randomness

Random sampling estimates $\pi$. The error shrinks as $1/\sqrt{N}$ — slow, but the method handles high dimensions and irregular domains where grid rules choke.

In [ ]:
from cds.montecarlo import estimate_pi

print(f"{'Samples':>12} {'Estimate':>12} {'Error':>12} {'Std-err':>12}")
print("-" * 50)
for n in [1_000, 10_000, 100_000]:
    result = estimate_pi(n_samples=n, seed=42)
    print(
        f"{n:>12,} {result.estimate:>12.6f} {result.estimate - math.pi:>+12.2e} {result.std_error:>12.2e}"
    )

## 3 · Linear Algebra — determinants scale as $O(N^3)$

PLU decomposition computes the determinant. Doubling $N$ should multiply the cost by $\approx 2^3 = 8$ for an $O(N^3)$ algorithm. Let's verify the complexity claim empirically.

In [ ]:
import time

from cds.math_utils import determinant, identity


def make_matrix(n):
    m = identity(n)  # start from identity (a list of lists)
    for i in range(n):  # perturb so the determinant is non-trivial
        m[i] = [1.0 / (i + j + 1) if i != j else float(n) for j in range(n)]
    return m


def time_det(n):
    m = make_matrix(n)
    t0 = time.perf_counter()
    determinant(m)
    return time.perf_counter() - t0


t50 = time_det(50)
t100 = time_det(100)
print(f"det(50x50)  = {t50 * 1000:>8.2f} ms")
print(f"det(100x100)= {t100 * 1000:>8.2f} ms")
print(f"Ratio (doubling N): {t100 / t50:.1f}x   (expected ≈ 8x for O(N³))")

## 4 · Signal Processing — FFT vs naive DFT

The Discrete Fourier Transform is $O(N^2)$. The Fast Fourier Transform (Cooley-Tukey, radix-2) is $O(N \log N)$. For $N=1024$ that's the difference between ~1M and ~10K operations.

In [ ]:
from cds.signals import dft, fft_radix2, power_spectrum

# A signal: two sine waves
N = 1024
signal = [
    math.sin(2 * math.pi * 5 * k / N) + 0.5 * math.sin(2 * math.pi * 13 * k / N) for k in range(N)
]

t0 = time.perf_counter()
naive = dft(signal)
t_dft = time.perf_counter() - t0
t0 = time.perf_counter()
fast = fft_radix2(signal)
t_fft = time.perf_counter() - t0

print(f"Naive DFT:  {t_dft * 1000:>8.2f} ms  (O(N²))")
print(f"FFT radix-2:{t_fft * 1000:>8.2f} ms  (O(N log N))")
print(f"Algorithmic speedup: {t_dft / t_fft:.0f}x faster")

mag = power_spectrum(signal)
peak_bin = max(range(len(mag)), key=lambda i: mag[i])
print(f"Dominant frequency bin: {peak_bin}  (matches the 5-cycle input)")

## 5 · Ordinary Differential Equations — the harmonic oscillator

Solve $y'' + y = 0$ (a pure cosine) as a system. Compare a first-order method (Euler) to a fourth-order one (RK4) at the same step size — Euler drifts, RK4 stays glued to the true solution.

In [ ]:
from cds.diffeq import solve_system


# System: y'' = -y  →  state vector [y, y'] with derivative [y', -y]
def harmonic(t, y):
    return [y[1], -y[0]]


t0, y0, t_end, h = 0.0, [1.0, 0.0], 6.283, 0.1  # one full period

# solve_system uses RK4 internally — true solution is cos(t), so y[0]≈1 at t=2π
ts, ys = solve_system(harmonic, t0, y0, t_end, dt=h)
rk4_final = ys[-1][0]

# Crude Euler for comparison: hand-rolled, same step size
y = list(y0)
for i in range(int(round((t_end - t0) / h))):
    deriv = harmonic(0.0, y)
    y = [yj + dj * h for yj, dj in zip(y, deriv)]
euler_final = y[0]

true_final = math.cos(t_end)
print(f"Euler final-position error: {euler_final - true_final:+.2e}  (drifts away)")
print(f"RK4   final-position error: {rk4_final - true_final:+.2e}  (stays on track)")
print(
    f"RK4 is ~{abs((euler_final - true_final) / (rk4_final - true_final)):,.0f}x more accurate at the same step size"
)

## 6 · Optimization — gradient descent on a bowl

Minimize $f(x,y) = (x-3)^2 + (y+1)^2$, whose minimum is at $(3, -1)$. Gradient descent follows the slope downhill.

In [ ]:
from cds.optimization import gradient_descent


def bowl(coords):
    x, y = coords
    return (x - 3) ** 2 + (y + 1) ** 2


result = gradient_descent(bowl, [0.0, 0.0])
print(f"Found minimum at {tuple(round(v, 6) for v in result.x)}")
print("True minimum at    (3.0, -1.0)")
print(f"Function value f* = {result.value:.2e}  (should be ≈ 0)")
print(f"Converged in {result.iterations} iterations: {result.converged}")

## 7 · Quantum — superposition & interference

Apply a Hadamard gate to $|0\rangle$ to create an equal superposition, then sample measurements. Each gate below has a small, readable implementation you can find in `src/cds/quantum/`.

In [ ]:
from cds.quantum import QuantumCircuit, hadamard, simulate

circuit = QuantumCircuit().add(hadamard())  # |0⟩ → (|0⟩ + |1⟩)/√2
result = circuit.run()
probs = result.probabilities()
print(f"Probabilities: |0⟩ → {probs[0]:.3f}, |1⟩ → {probs[1]:.3f}")

counts = simulate(circuit, shots=10000)
print(f"Measurement (10k shots): {counts}")
print("Each outcome lands near 50% — the hallmark of superposition.")

## 8 · Two-Dimensional Integration — the tensor product, and the curse of dimension

A 1-D rule extends to a rectangle by the **tensor product**: integrate in $x$ with a rule, then integrate *that result* in $y$. The catch is that work multiplies too — an $n	imes n$ grid costs $n^2$ evaluations, and in $d$ dimensions a full grid needs $n^d$. So, just as in stop 1, *high-order* rules pay off: a few well-placed points beat many evenly-spaced ones.

In [ ]:
import math
from cds.numerical_integration import simpson_2d, gaussian_quadrature_2d

# ∬_{[0,1]²} x² y² dx dy = (1/3)·(1/3) = 1/9 — exact for both rules.
def poly(x, y):
    return x * x * y * y
true_poly = 1.0 / 9.0

# Area of the unit disk ∬_{[-1,1]²} 1_{x²+y²≤1} dx dy should be π.
def disk(x, y):
    return 1.0 if x * x + y * y <= 1.0 else 0.0

print(f"Simpson 2-D  poly err = {simpson_2d(poly, 0,1, 0,1, 2,2)            - true_poly:+.2e}")
print(f"Gauss-3 2-D poly err = {gaussian_quadrature_2d(poly, 0,1, 0,1, n=3) - true_poly:+.2e}")
print(f"Simpson 2-D  disk    = {simpson_2d(disk, -1,1, -1,1, 100,100):.5f}  (π = 3.14159)")

**Takeaway:** both rules nail the smooth polynomial exactly (Simpson through cubics per axis, 3-point Gauss through degree 5). But the *disk* — a discontinuous indicator — still only reaches three decimals of $\pi$ from a $100	imes100 = 10{,}000$-point grid, because evaluations straddling the curved edge are wasted. As dimensions grow, the tensor-product grid explodes as $n^d$ — the precise pain Monte Carlo (stop 2) sidesteps.

In [ ]:
# See it explode: evaluations needed for a 2-D grid vs a 3-D grid at the same per-axis n.
for d in (1, 2, 3, 4, 5):
    print(f"d={d}: n=20 per axis -> {20**d:>12,} evaluations")

## 9 · Time-Series — structure in the order

Strip away the time index and a series is just numbers. Put it back, and *order* starts to matter: yesterday's value predicts today's. The **autocorrelation function** (ACF) measures exactly how much, at every lag, and the Ljung–Box test asks whether *any* of those correlations are real signal rather than noise.

In [ ]:
import random
from cds.stats import autocorrelation_function, ljung_box

random.seed(0)
N = 240
signal = [random.gauss(0.0, 1.0) for _ in range(N)]   # white-noise base...
for k in range(4, N):                                  # ...plus a lag-4 echo:
    signal[k] += 0.6 * signal[k - 4]                   # each value repeats 4 steps back

acf = autocorrelation_function(signal, max_lag=16)
print(f"r[1] = {acf[1]:+.3f}   (no lag-1 structure)")
print(f"r[4] = {acf[4]:+.3f}   (lag-4 echo peaks here)")
print(f"r[8] = {acf[8]:+.3f}   (echo of the echo)")
lb = ljung_box(signal, lags=12)
print(f"Ljung-Box p-value = {lb.p_value:.2e}  -> autocorrelation present? {lb.has_autocorrelation}")

**Takeaway:** the ACF is flat near zero *except* at lag 4 — exactly where we injected the echo — with a smaller secondary bump at lag 8 (each echo repeats its own predecessor). The Ljung–Box test aggregates the first dozen autocorrelations into one statistic and finds overwhelming evidence they are not all zero. This is step one of any Box–Jenkins workflow: detect structure, then model it. CDS also offers the partial ACF, KPSS-style stationarity testing, and seasonal decomposition.

In [ ]:
# Differencing at the echo lag kills the autocorrelation -> the series becomes white-ish noise.
from cds.stats import difference
diffed = difference(signal, lag=4)
acf_d = autocorrelation_function(diffed, max_lag=16)
print(f"after differencing: r[4] = {acf_d[4]:+.3f}   (was {acf[4]:+.3f})")

## 10 · Filter Design — maximally flat, from scratch

A filter's job is to pass some frequencies and stop others. The **Butterworth** family does this with the flattest possible passband — no ripple — which is why it's the default "honest" filter. Designing one means placing poles on a circle (analog prototype), warping them to the digital domain (bilinear transform), and reading off difference-equation coefficients — all derived from first principles in CDS.

In [ ]:
import math
from cds.signals import butter_lowpass, apply_filter

# A slow 5-cycle sine plus faster 200-cycle interference, sampled at N points.
# Normalised freq = (cycles) / (N/2): k=5 -> 0.010, k=200 -> 0.391.
N = 1024
clean = [math.sin(2*math.pi*5*k/N)   for k in range(N)]
high  = [0.6*math.sin(2*math.pi*200*k/N) for k in range(N)]
noisy = [a + b for a, b in zip(clean, high)]

coef = butter_lowpass(order=4, cutoff=0.15)     # keep freqs below 15% of Nyquist
filtered = apply_filter(noisy, coef)

# Cosine similarity in the steady-state half (phase-robust): 1.0 = perfect recovery.
half = N // 2
dot  = sum(clean[i]*filtered[i] for i in range(half, N))
norm = math.sqrt(sum(c*c for c in clean[half:]) * sum(f*f for f in filtered[half:]))
print(f"recovery (cos sim vs clean) = {dot/norm:.3f}   (1.000 = perfect)")
# Attenuation of the interference alone: pass it through and compare amplitudes.
attenuated = apply_filter([math.sin(2*math.pi*200*k/N) for k in range(N)], coef)
a_in  = 1.0
a_out = max(abs(x) for x in attenuated[half:])
print(f"interference amplitude: {a_in:.3f} -> {a_out:.4f}  ({a_in/max(a_out,1e-12):.0f}x smaller)")

**Takeaway:** the slow signal passes through almost untouched (cosine similarity $pprox 0.99$), while the 200-cycle interference is crushed ~74× ($-37$ dB). That selectivity comes from a 4th-order polynomial whose poles sit on a circle. The same module adds high-pass, band-pass (cascade), band-stop (parallel), and a `moving_median` denoiser that shrugs off impulsive outliers the way no linear filter can.

---

## 🎓 Where next?

Every algorithm above is **implemented from scratch** in `src/cds/` — readable Python you can step through line by line. To go deeper:

- 📖 [Full API reference](https://Furox88.github.io/cognitive-discovery-system/api/)
- 🎓 [Per-module tutorials](https://Furox88.github.io/cognitive-discovery-system/tutorials/quick_start/)
- 🔬 [Case study: measuring the Hubble constant](https://Furox88.github.io/cognitive-discovery-system/CASE_STUDY_HUBBLE/)

If this tour was useful, a ⭐ on the [repo](https://github.com/Furox88/cognitive-discovery-system) helps others find it.

> **Experiment:** change the function in §1, raise $N$ in §2, or add gates in §7. Nothing here is locked — that's the point of pure, readable code.